In [1]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

PyTorch : 2.12.0.dev20260408+cu128
CUDA    : True
GPU     : NVIDIA GeForce RTX 5080


In [2]:
from dotenv import load_dotenv
import os, hashlib, time, requests

load_dotenv()  # loads .env from current directory

API_KEY    = os.getenv("PODCASTINDEX_API_KEY")
API_SECRET = os.getenv("PODCASTINDEX_API_SECRET")

assert API_KEY and API_SECRET, "Missing credentials — check your .env file"

def auth_headers():
    epoch = str(int(time.time()))
    hash_input = API_KEY + API_SECRET + epoch
    auth_token = hashlib.sha1(hash_input.encode("utf-8")).hexdigest()
    return {
        "X-Auth-Key":    API_KEY,
        "X-Auth-Date":   epoch,
        "Authorization": auth_token,
        "User-Agent":    "MyPodcastApp/1.0",
    }

h = auth_headers()
print("Key:   ", h["X-Auth-Key"][:6], "...")
print("Date:  ", h["X-Auth-Date"])
print("Token: ", h["Authorization"][:12], "...")

Key:    SCHHAE ...
Date:   1778213466
Token:  8e71401049a2 ...


In [3]:
BASE_URL = "https://api.podcastindex.org/api/1.0"

def pi_get(endpoint, params=None):
    """Make an authenticated GET request and return the parsed JSON."""
    url = f"{BASE_URL}{endpoint}"
    resp = requests.get(url, headers=auth_headers(), params=params)
    resp.raise_for_status()
    return resp.json()

In [4]:
SHOW_NAME    = "Alex & Sigges Podcast"

# Step 1: find the show
shows = pi_get("/search/byterm", {"q": SHOW_NAME, "max": 5})
for i, f in enumerate(shows["feeds"]):
    print(f"[{i}] {f['title']}  (id={f['id']})")

[0] Alex & Sigges podcast  (id=720307)
[1] Alex & Sigges podcast  (id=5895478)


In [5]:
FEED_ID  = 720307  # or 5895478 — try both, one is likely a duplicate/mirror feed

episodes = pi_get("/episodes/byfeedid", {"id": FEED_ID, "max": 10})

for i, ep in enumerate(episodes["items"]):
    print(f"[{i}] {ep['title']}")
    print(f"     {ep['datePublishedPretty']}  |  {ep.get('duration',0)//60} min")
    print()

for f in shows["feeds"]:
    print(f"id={f['id']}  episodes={f.get('episodeCount','?')}  last_update={f.get('lastUpdateTime','?')}")
    print(f"  url={f.get('url')}")
    print()

[0] 728. Vänd dig inte om
     April 30, 2026 8:34pm  |  57 min

[1] 727. Flärpen
     April 23, 2026 6:44pm  |  53 min

[2] 726. Knäckgängensson, Skolsson, Frihetsson!
     April 16, 2026 5:57pm  |  53 min

[3] Extraavsnitt: Roligaste 2025
     April 09, 2026 10:07pm  |  67 min

[4] Extraavsnitt: Roligaste 2024
     April 02, 2026 6:06pm  |  57 min

[5] Extraavsnitt: Samlade hundliv
     March 26, 2026 7:19pm  |  71 min

[6] Extraavsnitt: Sigges gubbar
     March 19, 2026 7:26pm  |  76 min

[7] 725. Snickesnack, lek, test eller kärna?
     March 13, 2026 12:32am  |  58 min

[8] 724. Fem tavlor
     March 05, 2026 6:35pm  |  50 min

[9] 723. Mellan tårna
     February 26, 2026 4:18pm  |  61 min

id=720307  episodes=733  last_update=1777773220
  url=https://feed.pod.space/alexosigge

id=5895478  episodes=0  last_update=1670532117
  url=https://feeds.buzzsprout.com/2096542.rss



In [6]:
EPISODE_TITLE = "724. Fem tavlor"  # partial match

episodes_all = pi_get("/episodes/byfeedid", {"id": FEED_ID, "max": 733})

matches = [
    ep for ep in episodes_all["items"]
    if EPISODE_TITLE.lower() in ep["title"].lower()
]

for ep in matches:
    print(f"{ep['title']}")
    print(f"  {ep['datePublishedPretty']}  |  {ep.get('duration',0)//60} min")
    print(f"  {ep.get('enclosureUrl')}")

724. Fem tavlor
  March 05, 2026 6:35pm  |  50 min
  https://media.pod.space/alexosigge/aos724.mp3


In [1]:
import whisper, requests, io, time

AUDIO_URL = "https://media.pod.space/alexosigge/aos724.mp3"

print("Fetching audio...")
with requests.get(AUDIO_URL, stream=True) as r:
    total    = int(r.headers.get("content-length", 0))
    buf      = io.BytesIO()
    received = 0
    for chunk in r.iter_content(chunk_size=1024 * 1024):
        buf.write(chunk)
        received += len(chunk)
        print(f"\r  {received/1e6:.1f} / {total/1e6:.1f} MB", end="")

with open("temp_episode.mp3", "wb") as f:
    f.write(buf.getvalue())

print("\nLoading model...")
model = whisper.load_model("medium", device="cuda")

print("Transcribing...")
start  = time.time()
result = model.transcribe("temp_episode.mp3", language="sv", fp16=True)
elapsed = time.time() - start

print(f"\nDone in {elapsed/60:.1f} min")
print(result["text"][:1000])

Fetching audio...
  56.0 / 56.0 MB
Loading model...
Transcribing...

Done in 5.3 min
 Det här är Alex och Sigge och nu ska ni snart få lyssna på det avsnitt som vi släppte på podden för fyra veckor sen. Men som nu släpps i gratisfiden med reklam. Ja, jag hoppas att ni känner er trygga med det. Nöjda med det. För trygga är det väl. Men det kan vara på sin plats att också berätta om man vill ha avsnitten direkt. Så vi har precis berättat i ett avsnitt där jag berättade för dig om mitt möte med det just nu levande största geniet som finns i Sverige. Just det, och jag berättade om mitt möte med prinsessan Madeleine och om filmen Michael av Michael Jackson. Så det här är ju rykande färska grejer om man nu går in på podden och blir medlemmar. Intressant att du i ditt möte med Madeleine gjorde bort dig och jag i mitt möte med geniet triunferade. Så kan man säga. Ja. Allt det här kan ni lyssna på om ni går in på podden med appen. Och om ni inte gör det kommer den hamna i den här videon om fyra